In [ ]:
from pathlib import Path
from datetime import datetime
import os
import sys
project_root = Path('c:/PdM').resolve()
sys.path.insert(0, str(project_root))
from gas_turbine import GasTurbine
from physics.weather_api_client import create_hybrid_environment
from physics.environmental_conditions import EnvironmentalConditions, LocationType
from ml_utils.ml_output_modes import OutputMode
from datetime import datetime, timedelta

# Create hybrid environment with real weather and synthetic fallback

fallback = EnvironmentalConditions(LocationType.TROPICAL)
env_source = create_hybrid_environment(
    use_real_weather=True,
    api_provider="weatherapi",
    api_key = os.getenv('WEATHER_API_KEY'),
    location_name="Port Harcourt",  # Nigerian coastal installation
    country="Nigeria",
    fallback_source=fallback,  # Falls back to synthetic Tropical on API failure
    cache_enabled=True
)

turbine = GasTurbine(
    name='GT-PH-001',
    initial_health={'hgp': 0.85, 'blade': 0.90, 'bearing': 0.80, 'fuel': 0.88},
    # Pass real weather directly via env_model parameter
    env_model=env_source,  # Real weather from Port Harcourt, Nigeria
    enable_enhanced_vibration=True,
    enable_thermal_transients=True,
    enable_environmental=True,
    enable_maintenance=True,
    enable_incipient_faults=True,
    enable_process_upsets=True,
    output_mode=OutputMode.FULL
)

turbine.set_speed(9500)

# Simulate 30 days with real weather
start_time = datetime(2026, 1, 1)
for hour in range(30 * 24):  # 30 days, hourly
    timestamp = start_time + timedelta(hours=hour)

    # Real weather is automatically fetched and cached
    state = turbine.next_state()

    # Log real weather conditions
    if hour % 24 == 0:  # Daily log
        day = hour // 24
        print(f"Day {day}: Temp={state.get('ambient_temp', 'N/A')}°C, "
              f"EGT={state['exhaust_gas_temp']:.1f}°C, "
              f"Efficiency={state['efficiency']:.3f}")

In [ ]:
from gas_turbine import generate_turbine_dataset

telemetry, failures = generate_turbine_dataset(
    n_machines=5,              # 10 turbines
    n_cycles_per_machine=100,   # 100 operating cycles each
    cycle_duration_range=(60, 300),  # 1-5 minutes per cycle
    random_seed=42
)

print(f"Generated {len(telemetry)} telemetry records")
print(f"Captured {len(failures)} failures")

# Failure distribution
for failure in failures:
    print(f"{failure['machineID']}: {failure['code']} - {failure['message']}")

In [33]:
from gas_turbine import GasTurbine
from ml_utils.ml_output_modes import OutputMode
import pandas as pd

# Training data (with ground-truth health)
turbine_train = GasTurbine('GT-Train', output_mode=OutputMode.DELAYED_LABELS)
train_data = []

for i in range(10000):
    state = turbine_train.next_state()
    # state includes 'health_hgp', 'health_blade', etc.
    train_data.append(state)

train_data = pd.DataFrame(train_data)

train_data

,speed,speed_target,exhaust_gas_temp,oil_temp,fuel_flow,vibration_rms,vibration_peak,compressor_discharge_temp,compressor_discharge_pressure,ambient_temp,ambient_pressure,efficiency,operating_hours,health_hgp,health_blade,health_bearing,health_fuel
0,-2.66,0.0,25.15,25.46,-0.007,0.0,0.0,25.38,99.04,25.01,101.22,1.0,0.0,0.9200,0.9500,0.9000,0.93
1,-7.54,0.0,24.10,24.98,0.053,0.0,0.0,26.13,107.09,25.05,101.24,1.0,0.0,0.9200,0.9500,0.9000,0.93
2,-3.93,0.0,23.58,24.60,0.020,0.0,0.0,23.70,100.38,24.87,101.24,1.0,0.0,0.9200,0.9500,0.9000,0.93
3,-3.28,0.0,25.05,24.41,0.054,0.0,0.0,25.14,101.54,24.97,101.27,1.0,0.0,0.9200,0.9500,0.9000,0.93
4,-0.61,0.0,23.44,24.78,-0.020,0.0,0.0,24.36,98.69,25.19,101.30,1.0,0.0,0.9200,0.9500,0.9000,0.93
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,3.40,0.0,24.27,25.46,-0.036,0.0,0.0,23.38,101.00,24.86,101.35,1.0,0.0,0.9185,0.9496,0.8717,0.93
9996,9.70,0.0,28.00,25.34,0.108,0.0,0.0,24.96,93.47,24.98,101.31,1.0,0.0,0.9185,0.9496,0.8716,0.93
9997,-1.91,0.0,24.54,24.94,0.032,0.0,0.0,26.52,106.11,25.14,101.28,1.0,0.0,0.9185,0.9496,0.8716,0.93
9998,3.98,0.0,25.50,24.82,0.060,0.0,0.0,25.47,104.46,25.16,101.34,1.0,0.0,0.9185,0.9496,0.8716,0.93
